# Label extraction from radiology reports

Select the **Python (RSNA Knee)** kernel in the top-right before running cells.

Reports exist in train only. The MRI model never sees them at test time. This notebook builds a **report → 12 binary labels** mapper so the ~4,349 unlabeled studies can become silver training data.

**v0 plan (do in order):**
1. Harness: `predict` + `evaluate` on gold (this notebook inits that).
2. Keyword rules, starting with Effusion / Baker's, then ACL (watch negation).
3. Error analysis: print false positives / false negatives for the worst label.
4. Fix one failure class, re-score. Only then consider a heavier model.

Your work starts at the keyword dict inside `predict`. Run the all-zero baseline first so you know the scoreboard works (recall should be 0).

In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Working directory:", Path.cwd())

Python: 3.11.5
pandas: 3.0.5
Working directory: /Users/paramtully/IdeaProjects/TechnicalProjects/RSNA-Knee-Abnormality-Detection


In [4]:
DATA_DIR = Path(".")

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

train = pd.read_csv(DATA_DIR / "train.csv")
gold = train.loc[train[LABELS].notna().all(axis=1)].copy()
unlabeled = train.loc[train[LABELS].isna().all(axis=1)].copy()

print("train:", train.shape)
print("gold:", gold.shape)
print("unlabeled:", unlabeled.shape)
assert gold.shape[0] == 58, gold.shape[0]
assert unlabeled.shape[0] == 4407 - 58, unlabeled.shape[0]

train: (4407, 14)
gold: (58, 14)
unlabeled: (4349, 14)
